# FinAccess 2021 Microdata Exploration

This note book explores the srtucture of the FinAccess 2021 Microdata Excel Workbook, inspects variables, creates a variable dictionary, and identifies possible target variables for financial inclusion analysis.


# Abstract

This project focuses on predicting exclusion risk among Kenyan adults using machine learning and explainable AI techniques. Although Kenya is widely known for mobile money, many people still lack access to formal finacial services such as banks, insurance, SACCOs, pensions and regulated credit products. The goal of this project is to identifiy individuals who are most at risk of financial exclusion, understand the main factors contributing to that risk, and analyze how exclusion, patterns vary across counties and vulnerable groups in Kenya.

The project will use the 2021 FinAccess Household Survey microdata as the maiin dataset, while the 2024 FinAccess report and dashboard will be used for comparison and interpretation of trends. Several machine learning models, including Logistic Regression, Decision Trees, Random Forest,Gradient Boosting, and XGBoost. will be explored and evalluated using performance metrics such as accuracy, precision, recall, F1-score, and ROC-AUC. SHAP explainability techniques will also be used to show the key factors influencing financial exclusion predictions.

In addition to predictive modeling, the project will include county-level risk mapping, sub-group analysis, and the development of a Flask web application that provides financial exclusion probability scores, risk tiers, and key contributing factors. The findings from this project cold help banks, fintech companies, Saccos, policy makers, and development organizations design more targeted financial inclusion strategies and interventions across kenya.

In [2]:
import pandas as pd 
import openpyxl 
import os

print(openpyxl.__version__)
print(os.listdir())


# Load the Excel file
excel_file = pd.ExcelFile("../data/raw/finaccess_2021_microdata.xlsx")

# View sheet names
print(excel_file.sheet_names)

# Load the first sheet
df = pd.read_excel("../data/raw/finaccess_2021_microdata.xlsx", sheet_name="Dataset",nrows=1000,engine="openpyxl")
df.head()

ImportError: DLL load failed while importing _multiarray_umath: The specified module could not be found.

3.1.5
['01_data _understanding.ipynb', 'dummy.ipynb.txt']
['Dataset', 'Variable Information', 'Variable Values']


,Serial Number,County,ClusterNo,HHNo,interview__key,interview__id,A9,A9i,A10i,A14v,...,allotherformal_banked2022,formal_banked2022,excluded_informal_banked2022,NHIF_health_insurance,NHIF_ONLY,Medical_Insurance_ONLY,BothNHIF_medical,Nomedical,NHIFMedical_Cat,adults
0,1,Trans Nzoia,10226038,1048,10-67-89-46,0003fc74b3fe418ea041bd6a9e7ff387,Rural,Access granted,Female,1,...,Yes,No,Other Formal,Yes,NaN,NaN,1.0,NaN,Users of NHIF + medical insurance (C1_42 and C...,1 adult Household
1,2,Busia,10240034,1080,39-64-68-81,0004890b17744272baf0a0c7b4c20771,Rural,Access granted,Female,4,...,No,Yes,Banked,Yes,1.0,NaN,NaN,NaN,"Users of NHIF only, with no medical insurance",>1 adult Household
2,3,Machakos,10216062,1013,92-34-74-01,00052153fe8c4abaa189caadcb87b2b4,Rural,Access granted,Male,1,...,No,No,Excluded,Yes,1.0,NaN,NaN,NaN,"Users of NHIF only, with no medical insurance",1 adult Household
3,4,Kisumu,10242078,1026,08-14-22-63,000d1f8747194b6a84862830dc5fe7ca,Rural,Access granted,Male,5,...,No,Yes,Banked,No,NaN,NaN,NaN,"Users of NHIF only, with no medical insurance",None users of any of NHIF and medical insurance,>1 adult Household
4,5,Nyeri,10219138,1019,99-12-05-84,000f5a5c0e3246ac9a62603ad936e3da,Urban,Access granted,Male,3,...,No,Yes,Banked,No,NaN,NaN,NaN,"Users of NHIF only, with no medical insurance",None users of any of NHIF and medical insurance,>1 adult Household


In [ ]:
df.shape

(1000, 2332)

In [ ]:
df.columns

Index(['Serial Number', 'County', 'ClusterNo', 'HHNo', 'interview__key',
       'interview__id', 'A9', 'A9i', 'A10i', 'A14v',
       ...
       'allotherformal_banked2022', 'formal_banked2022',
       'excluded_informal_banked2022', 'NHIF_health_insurance', 'NHIF_ONLY',
       'Medical_Insurance_ONLY', 'BothNHIF_medical', 'Nomedical',
       'NHIFMedical_Cat', 'adults'],
      dtype='object', length=2332)

In [ ]:
df.describe()

,Serial Number,ClusterNo,HHNo,A14v,A14vi,A17,A19,A21i,B3Di.2,B3Di.3,...,non_Informal_only_credit_users2,NotFormal_non_digital_only,all_loans,received1,sent1,loan_Traditional,informal_sector,NHIF_ONLY,Medical_Insurance_ONLY,BothNHIF_medical
count,1000.000000,1.000000e+03,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,0.0,0.0,0.0,...,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,155.0,25.0,6.0
mean,500.500000,1.020131e+07,1053.782000,2.39000,1.754000,0.097000,38.907000,NaN,NaN,NaN,...,0.241000,0.488000,0.37200,0.915000,0.844000,0.031000,0.660000,1.0,1.0,1.0
std,288.819436,4.528596e+04,36.683404,1.40637,1.216126,0.517551,16.948971,NaN,NaN,NaN,...,0.427904,0.500106,0.48358,0.279021,0.363037,0.173404,0.473946,0.0,0.0,0.0
min,1.000000,1.010103e+07,1001.000000,1.00000,1.000000,0.000000,16.000000,NaN,NaN,NaN,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,1.0,1.0,1.0
25%,250.750000,1.020110e+07,1025.000000,1.00000,1.000000,0.000000,26.000000,NaN,NaN,NaN,...,0.000000,0.000000,0.00000,1.000000,1.000000,0.000000,0.000000,1.0,1.0,1.0
50%,500.500000,1.021806e+07,1048.000000,2.00000,1.000000,0.000000,35.000000,NaN,NaN,NaN,...,0.000000,0.000000,0.00000,1.000000,1.000000,0.000000,1.000000,1.0,1.0,1.0
75%,750.250000,1.023439e+07,1075.000000,3.00000,2.000000,0.000000,49.000000,NaN,NaN,NaN,...,0.000000,1.000000,1.00000,1.000000,1.000000,0.000000,1.000000,1.0,1.0,1.0
max,1000.000000,1.024729e+07,1236.000000,10.00000,11.000000,6.000000,91.000000,NaN,NaN,NaN,...,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.0,1.0,1.0


## Dataset sheet

This sheet contains the actual survey responses collected from respondents during he 2021 FinAccess survey.

In [7]:
# Load the second sheet
df = pd.read_excel("../data/raw/finaccess_2021_microdata.xlsx", sheet_name="Variable Information", nrows=1000, engine="openpyxl",header=1)
df.head()

,Variable,Position,Label,Column Width,Write Format,Missing Values
0,County,1,County,8,F12,NaN
1,ClusterNo,2,A7. NASSEP Cluster Number,18,F12,NaN
2,HHNo,3,A8. Household Number,18,F12,NaN
3,interview__key,4,Interview key (identifier in XX-XX-XX-XX format),33,A99,NaN
4,interview__id,5,Unique 32-character long identifier of the int...,50,A288,NaN


In [4]:
df.shape

(1000, 6)

In [5]:
df.columns

Index(['Variable Information', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3',
       'Unnamed: 4', 'Unnamed: 5'],
      dtype='object')

## Variable Information Sheet Description
This sheet provides definitions and descriptions for variables contained in the dataset sheet. It helps explain what each column represents.

## Important Variables Dictionary
The variable information sheet contains descriptions of variables used in the dataset. This section creates a variable dictionary to help interpret important variables and understand their meaning during analysis.

In [9]:
#Load Variable Information sheet
variable_info_df = pd.read_excel("../data/raw/finaccess_2021_microdata.xlsx",sheet_name="Variable Information",engine="openpyxl",header=1)

#create variable dictionary
variable_dict = dict(zip(variable_info_df["Variable"], variable_info_df["Label"]))

#display variable dictionary
variable_dict

{'County': 'County',
 'ClusterNo': 'A7. NASSEP Cluster Number',
 'HHNo': 'A8. Household Number',
 'interview__key': 'Interview key (identifier in XX-XX-XX-XX format)',
 'interview__id': 'Unique 32-character long identifier of the interview',
 'A9': 'A9. Cluster type',
 'A9i': 'A9i. What was the RESULTS of this interview?',
 'A10i': 'A10i. Sex',
 'A14v': 'A14v Number of eligible persons',
 'A14vi': 'A14vi Index of the selected person',
 'A15': 'A15. Language',
 'A15i': 'A15i. Language',
 'A17': 'A17. Not returned to complete their education',
 'A18': 'A17. Selected Respondent sex',
 'A19': 'A19. AGE OF RESPONDENT',
 'A20': 'A20. Relation to HH Head',
 'A21': 'A21. Education Completed',
 'A21i': 'A21i. Other Education level Completed',
 'A22': 'A22. Marital Status',
 'A23': 'A23. Female HH Highest Formal Education',
 'A24': 'A24. Chronic Disease',
 'A25': 'A25. Persons with disability',
 'A26__1': 'A26. Persons with disability:Difficulty in seeing',
 'A26__2': 'A26. Persons with disabili

In [13]:
# Load the third sheet
df = pd.read_excel("../data/raw/finaccess_2021_microdata.xlsx", sheet_name="Variable Values",nrows=1000, engine="openpyxl",header=1)
df.head()

,Value,Unnamed: 1,Label
0,County,1,Mombasa
1,NaN,2,Kwale
2,NaN,3,Kilifi
3,NaN,4,Tana River
4,NaN,5,Lamu


In [14]:
df.shape

(1000, 3)

In [15]:
df.columns

Index(['Value', 'Unnamed: 1', 'Label'], dtype='object')

In [16]:
df.describe()

,Value,Unnamed: 1,Label
count,232,1000,1000
unique,232,63,269
top,County,1,No
freq,1,213,134


## Variable Values Sheet Description
This sheet explains the coded values used within variables. It maps numerical or categorical codes to their actual meanings.